## Sustainability & Spatial Misalignment in Early-Stage Corridor Development

| Hypothesis | Statement |
|---|---|
| H1 | Built-up expansion is associated with declining vegetation cover and increased surface heat susceptibility |
| H2 | A share of new built-up areas overlaps with environmentally sensitive or water-prone zones |
| H3 |  Areas with low economic utilisation disproportionately coincide with higher environmental exposure |

In [ ]:
import geemap
import ee 
import geemap.colormaps as cm
import os

In [ ]:
geemap.ee_initialize()

In [ ]:
# Define the Relative Path
# '..' means go up one folder from where this notebook is saved
file_path = os.path.join('..', 'data', 'processed', 'Dholera_Taluk.geojson')

Map = geemap.Map()

In [ ]:
roi = geemap.geojson_to_ee(file_path)
Map.addLayer(roi, {'color': 'blue', 'width': 2}, 'Dholera Taluka Boundary')
Map.centerObject(roi, 10)
Map

In [ ]:
# ── Sentinel-2: 2025 (Oct–Dec post-monsoon composite) ───────
s2_2025 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2025-10-01', '2025-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

# ── Sentinel-2: 2016 (Oct–Dec post-monsoon composite) ───────
s2_2016 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2016-10-01', '2016-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

Computes NDBI, MNDWI, SAVI, and NDVI for a given Sentinel-2 image.  
NDVI is added here (not in RQ1/RQ2) as it provides a cleaner vegetation-loss signal alongside SAVI.

In [ ]:
def calculate_indices(image):
    """
    Computes NDBI, MNDWI, SAVI, NDVI from a Sentinel-2 SR image.
    Band references:
        B2=Blue, B3=Green, B4=Red, B8=NIR, B11=SWIR1
    """
    ndbi = image.expression(
        '(SWIR1 - NIR) / (SWIR1 + NIR)',
        {'SWIR1': image.select('B11'), 'NIR': image.select('B8')}
    ).rename('NDBI')

    mndwi = image.expression(
        '(Green - SWIR1) / (Green + SWIR1)',
        {'Green': image.select('B3'), 'SWIR1': image.select('B11')}
    ).rename('MNDWI')

    savi = image.expression(
        '((NIR - Red) * 1.5) / (NIR + Red + 0.5)',
        {'NIR': image.select('B8'), 'Red': image.select('B4')}
    ).rename('SAVI')

    return image.addBands([ndbi, mndwi, savi])


processed_2025 = calculate_indices(s2_2025)
processed_2016 = calculate_indices(s2_2016)

print("Spectral indices computed for 2016 and 2025.")

---
## Built-up Masks (Reusing RQ1 thresholds)

| Year | NDBI threshold | Rationale |
|---|---|---|
| 2025 | > 0.05 | Post-monsoon; lower saline soil noise |
| 2016 | > 0.13 | Higher radiometric contrast from saline pans in drier 2016 season |

In [ ]:
# ── 2025 Built-up Mask ───────────────────────────────────────
built_up_2025 = processed_2025.expression(
    '(NDBI > 0.05) && (MNDWI < 0) && (SAVI < 0.18)',
    {
        'NDBI':  processed_2025.select('NDBI'),
        'MNDWI': processed_2025.select('MNDWI'),
        'SAVI':  processed_2025.select('SAVI')
    }
).rename('BuiltUp_2025')

# ── 2016 Built-up Mask ───────────────────────────────────────
built_up_2016 = processed_2016.expression(
    '(NDBI > 0.13) && (MNDWI < 0) && (SAVI < 0.18)',
    {
        'NDBI':  processed_2016.select('NDBI'),
        'MNDWI': processed_2016.select('MNDWI'),
        'SAVI':  processed_2016.select('SAVI')
    }
).rename('BuiltUp_2016')

print("Built-up masks ready: 2016 and 2025.")

---
### Land Transformation Map

A two-panel view of:
- **Built-up change** (2016 → 2025) — reusing the 4-class growth heatmap from RQ1
- **Vegetation loss** — NDVI/SAVI change between 2016 and 2025

Both layers are shown together so you can read **where built-up growth co-occurred with vegetation loss**.

Built-up Growth Heatmap (2016 → 2025)

| Value | Class | Colour |
|---|---|---|
| 0 | No built-up (both years) | ⬛ |
| 1 | Stable built-up | ⬜ |
| 2 | New growth 2016→2025 | 🟠 |
| 3 | Lost built-up | 🔵 |

In [ ]:
# ── Built-up Change Classification ──────────────────────────
change_stack = built_up_2016.rename('y2016').addBands(built_up_2025.rename('y2025'))

growth_class = change_stack.expression(
    "(b('y2016') == 0 && b('y2025') == 0) ? 0"   # No built-up
    ": (b('y2016') == 1 && b('y2025') == 1) ? 1"  # Stable
    ": (b('y2016') == 0 && b('y2025') == 1) ? 2"  # New Growth
    ": (b('y2016') == 1 && b('y2025') == 0) ? 3"  # Lost
    ": 0"
).rename('GrowthClass').clip(roi)

### Vegetation Loss Layer (NDVI change & SAVI change)

Negative delta = vegetation lost.  
We compute both NDVI delta and SAVI delta - SAVI is more reliable over Dholera's arid/saline background.

In [ ]:
# ── Vegetation Change Layers ─────────────────────────────────

# SAVI delta: more soil-noise-robust over arid landscapes
savi_2025 = processed_2025.select('SAVI')
savi_2016 = processed_2016.select('SAVI')
savi_delta = savi_2025.subtract(savi_2016).rename('SAVI_Delta').clip(roi)

print("Vegetation change layers computed.")

#### Visualising SAVI and change in SAVI (2025-2016)

In [ ]:

# --- Map 3: SAVI/NDBI (Testing)---
Map_Test = geemap.Map()
Map_Test.centerObject(roi, 10)
test_params = {'min': 0, 'max': 1, 'palette': ['8b0000', 'f5f5f5', '006400']}

Map_Test.addLayer(savi_2025, test_params, 'SAVI (2025)')
Map_Test.addLayer(savi_2016, test_params, 'SAVI (2016)')
Map_Test.add_colorbar(vis_params=test_params, label="Test Value", orientation="horizontal")
print("Displaying Test Map...")
display(Map_Test)

In [ ]:
# --- Delta Calculations ---
Map_Test = geemap.Map()
Map_Test.centerObject(roi, 10)
# SAVI Delta: (Later Year - Earlier Year)
# Positive values = Increase in vegetation/soil health
# Negative values = Decrease/Degradation
savi_delta = savi_2025.subtract(savi_2016)

# Define a specific Delta Palette (Red for decrease, White for neutral, Blue/Green for increase)
delta_params = {'min': -0.5, 'max': 0.5, 'palette': ['8b0000', 'f5f5f5', '006400']}

# Add to a new map or the existing one
Map_Test.addLayer(savi_delta, delta_params, 'SAVI Change (2016-2025)')

Map_Test.add_colorbar(vis_params=delta_params, label="Delta Value", orientation="horizontal")

print("Delta layers calculated and added.")
display(Map_Test)

#### OUTPUT 1 : Visualise: Land Transformation Map

In [ ]:
# ── Output 1: Built-up Transformation Map ────────────────────
# Logic: Growth Class == 2 (New Growth) AND SAVI Delta < 0
# This isolates pixels that urbanised AND lost vegetation simultaneously

# Step 1: Mask SAVI delta to new growth footprint only
new_growth_mask   = growth_class.eq(2)                        # Class 2 = new growth pixels
veg_loss_in_growth = savi_delta.updateMask(                   # Keep only pixels where:
    new_growth_mask                                            #   (a) new growth occurred
    .And(savi_delta.lt(0))                                     #   (b) SAVI declined
)

# Step 2: New growth footprint as a flat base layer (charcoal)
new_growth_flat = new_growth_mask.selfMask()

# ── Map ──────────────────────────────────────────────────────
Map_LT = geemap.Map()
Map_LT.centerObject(roi, 11)

# Reference
Map_LT.addLayer(
    s2_2025, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'S2 True Colour (2025)', shown=False
)

# Layer 1: Full new-growth footprint as flat charcoal base
# This shows the full 34 km² even where SAVI didn't decline
Map_LT.addLayer(
    new_growth_flat,
    {'min': 1, 'max': 1, 'palette': ['444444']},
    '⬛ New Growth Footprint (34 km²)'
)

# Layer 2: Vegetation loss gradient — overlaid WITHIN the new growth footprint
# Palette: pale yellow (marginal loss) → deep red (severe loss)
# min is set to -0.3 instead of -0.5 to stretch the colour ramp
# across the realistic SAVI loss range for Dholera's arid landscape
Map_LT.addLayer(
    veg_loss_in_growth,
   # {'min': -0.3, 'max': 0.0, 'palette': ['ffffb2', 'fd8d3c', 'e31a1c', '800026']},
    {'min': -0.3, 'max': 0.0, 'palette': ['800026','e31a1c','fd8d3c','ffffb2']},
    '🔴 Vegetation Loss Intensity (SAVI Decline in New Growth)'
)

# Colorbar
Map_LT.add_colorbar(
    vis_params={'min': -0.3, 'max': 0.0,
                'palette': ['800026','e31a1c','fd8d3c','ffffb2']},
    label='ΔSAVI (0 = No Loss  →  −0.3 = Severe Loss)',
    orientation='horizontal'
)

Map_LT.addLayer(
    roi.style(color='00ffcc', fillColor='00000000', width=2), {},
    'Dholera Boundary'
)

print("Output 1: Built-up Transformation Map ready.")
display(Map_LT)

### Vegetation Loss Area Statistics

Measures the area where SAVI *declined* (vegetation lost) within the new growth footprint - directly tests **H1**.

In [ ]:
# ── Step 3: Key Statistic — Average SAVI Loss per km² of New Growth ──
# Mean SAVI delta across the masked (new growth AND veg loss) zone
mean_savi_loss = savi_delta.updateMask(new_growth_mask.And(savi_delta.lte(0))) \
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10,
        maxPixels=1e13
    ).getInfo()

pixel_area = ee.Image.pixelArea()

# Area of new growth with confirmed veg loss
veg_loss_area_m2 = new_growth_mask.And(savi_delta.lt(0)) \
    .multiply(pixel_area) \
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=10,
        maxPixels=1e13
    ).getInfo()

avg_savi_loss   = round(mean_savi_loss.get('SAVI_Delta', 0), 4)
veg_loss_km2    = round(list(veg_loss_area_m2.values())[0] / 1e6, 3)
total_growth_km2 = 34.013   # from RQ1

print()
print("=" * 55)
print("  OUTPUT 1 — KEY STATISTICS")
print("=" * 55)
print(f"  🟠 New Built-up Growth (2016→2025)     : {total_growth_km2} km²")
print(f"  🌿 Growth ∩ Vegetation Loss Area        : {veg_loss_km2} km²")
if total_growth_km2 > 0:
    pct = round(veg_loss_km2 / total_growth_km2 * 100, 1)
    print(f"  → {pct}% of new growth occurred at cost of local biomass")
print(f"  📉 Avg SAVI Loss per km² of New Growth  : {avg_savi_loss}")
print()
print("  Ecological Interpretation:")
print("  Each km² of new industrial footprint carries an average SAVI")
print(f"  decline of {avg_savi_loss} - the ecological price tag of corridor expansion.")
print("=" * 55)

---
## OUTPUT 2 : Heat Susceptibility Map

> **Proxy for urban heat stress: high built-up density + low vegetation**

This is a **composite index** - not a direct LST measurement - built from:
- `NDBI` (built-up surface): higher = more heat-absorbing materials
- `SAVI` (vegetation): higher = more evapotranspiration, lower heat stress

**Formula:**
```
Heat_Index = NDBI_norm - SAVI_norm
```
Values range ~[-1 to +1].  
+ Positive = high heat susceptibility (built-up, no vegetation)  
- Negative = low heat susceptibility (vegetated, no built-up)

In [ ]:
# ── Heat Susceptibility Index (2025) ─────────────────────────
# Normalise NDBI and SAVI to 0–1 range before differencing
# Using min–max normalisation anchored to realistic Sentinel-2 ranges

ndbi_raw  = processed_2025.select('NDBI')
savi_raw  = processed_2025.select('SAVI')

# NDBI normalised: range −0.5 to +0.5
ndbi_norm = ndbi_raw.add(0.5).divide(1.0).clamp(0, 1).rename('NDBI_norm')

# SAVI normalised: range 0 to 0.8
savi_norm = savi_raw.divide(0.8).clamp(0, 1).rename('SAVI_norm')

# Heat Susceptibility = Built-up signal minus Vegetation buffer
heat_index = ndbi_norm.subtract(savi_norm).rename('HeatIndex').clip(roi)

print("Heat Susceptibility Index computed.")

In [ ]:
# ── Binary Heat Susceptibility Mask ──────────────────────────
# A pixel is flagged as heat-susceptible when:
#   NDBI > 0.05  (confirmed built-up surface)
#   SAVI < 0.18  (minimal vegetation buffer)
# This is the exact built-up condition from RQ1 - the built-up mask
# IS the heat susceptibility mask (all built-up is low-vegetation by definition).
# The Heat Index continuous raster is the gradient version for mapping.

heat_susceptible = built_up_2025.eq(1).rename('HeatSusceptible')

print("Binary heat-susceptible mask ready (= 2025 built-up footprint).")

In [ ]:
# ── Visualise: Heat Susceptibility Map ───────────────────────
Map_Heat = geemap.Map()
Map_Heat.centerObject(roi, 11)

# Reference
Map_Heat.addLayer(
    s2_2025, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'S2 True Colour (2025)', shown=False
)

# --- Layer 1: Continuous Heat Index (gradient) ---
# Palette: cool blue → white → fiery red
Map_Heat.addLayer(
    heat_index,
    {'min': -0.5, 'max': 0.7, 'palette': ['313695', '74add1', 'ffffbf', 'f46d43', 'a50026']},
    '🌡️ Heat Susceptibility Index (Continuous)'
)

# --- Layer 2: High-susceptibility zone only (built-up pixels) ---
# Shown as semi-transparent red overlay
Map_Heat.addLayer(
    heat_susceptible,
    {'min': 0, 'max': 1, 'palette': ['00000000', 'ff2200']},
    '🔴 High Heat Zone (Built-up, Low Veg)', shown=False
)

# --- Layer 3: SAVI 2025 for reference (vegetation buffer) ---
Map_Heat.addLayer(
    savi_2025,
    {'min': 0, 'max': 0.6, 'palette': ['ffffff', 'c8e6c9', '1b5e20']},
    'SAVI 2025 (Vegetation Reference)', shown=False
)

# Colorbar for heat index
Map_Heat.add_colorbar(
    vis_params={'min': -0.5, 'max': 0.7, 'palette': ['313695', '74add1', 'ffffbf', 'f46d43', 'a50026']},
    label='Heat Susceptibility Index  \n (−0.5 = Cool/Vegetated → +0.7 = Hot/Built-up)',
    orientation='horizontal'
)

Map_Heat.addLayer(
    roi.style(color='00ffcc', fillColor='00000000', width=2), {},
    'Dholera Boundary'
)

print("Heat Susceptibility Map ready.")
display(Map_Heat)

#### Heat Zone Area Statistics

In [ ]:
# ── Area Stats: Heat Susceptibility ──────────────────────────

# Helper
def area_km2(mask, band_name='BuiltUp'):
    result = mask.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=10,
        maxPixels=1e13
    )
    return round(result.getInfo().get(band_name, 0) / 1e6, 3)

# Recompute new_growth_pixels from growth_class (already in memory)
new_growth_pixels = growth_class.eq(2)

# High-heat zone = entire 2025 built-up footprint
heat_area_km2 = area_km2(heat_susceptible.rename('BuiltUp'), 'BuiltUp')

# NEW growth in 2025 that is heat-susceptible
new_growth_heat = new_growth_pixels.And(heat_susceptible)
new_growth_heat_km2 = area_km2(new_growth_heat.rename('BuiltUp'), 'BuiltUp')

print("=" * 55)
print("  HEAT SUSCEPTIBILITY REPORT (2025)")
print("=" * 55)
print(f"  🔴 Total High-Heat Zone (2025)     : {heat_area_km2:>8.3f} km²")
print(f"  🟠 New Growth in High-Heat Zone    : {new_growth_heat_km2:>8.3f} km²")
if heat_area_km2 > 0:
    pct_new = round(new_growth_heat_km2 / heat_area_km2 * 100, 1)
    print(f"  → {pct_new}% of the heat-susceptible footprint is from post-2016 growth.")
print()
print("  Interpretation:")
print("  Infrastructure-led expansion is the primary driver of heat")
print("  exposure — replacing bare/saline land with impervious surfaces")
print("  without compensatory green cover.")
print("=" * 55)